In [2]:
import os
from google.colab import userdata

# Retrieve your Groq API Key from Colab's left sidebar Secrets menu (🔑 icon) named "GROQ_API_KEY"
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API')

In [4]:
!pip install langgraph langchain-groq pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.8 MB/s eta 0:00:00


In [9]:
import os
import json
from typing import TypedDict, Optional, List
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

# ==========================================
# 1. DEFINE AGENT STATE & STRUCTURED SCHEMA
# ==========================================

class FinancialProfile(BaseModel):
    income: Optional[float] = Field(None, description="Monthly income or allowance in Rupees")
    expenses: Optional[float] = Field(None, description="Average monthly expenses in Rupees")
    reason_for_investment: Optional[str] = Field(None, description="The core goal or reason for wanting to invest")
    missing_question: Optional[str] = Field(None, description="A single polite question if any of the above 3 fields are missing. If none are missing, leave empty.")

class AgentState(TypedDict):
    messages: List[BaseMessage]
    profile: FinancialProfile
    initial_recommendation: Optional[str]
    current_step: str

UPGRADED_MODEL = "llama-3.3-70b-versatile"
llm = ChatGroq(model=UPGRADED_MODEL, temperature=0.2)

# ==========================================
# 2. DEFINE THE NODES (OPERATIONAL BLOCKS)
# ==========================================

def gather_info_node(state: AgentState):
    """Steps 1, 2 & 3: Extracts profile variables and checks for missing entries."""
    messages = state["messages"]

    system_prompt = (
        "You are an information extraction assistant. Analyze the chat history between the user and the agent. "
        "Extract: income, expenses, and reason_for_investment.\n"
        "If any of these three fields are missing, generate a polite question to ask for ONE missing piece of data in the 'missing_question' field.\n"
        "If all three pieces of data are present, set 'missing_question' to empty or null."
    )

    structured_llm = llm.with_structured_output(FinancialProfile)
    updated_profile = structured_llm.invoke([AIMessage(content=system_prompt)] + messages)

    next_step = "VERIFICATION" if not updated_profile.missing_question else "GATHERING"
    return {"profile": updated_profile, "current_step": next_step}


def recommend_node(state: AgentState):
    """Step 4: Generates a highly practical, relatable high-level allocation recommendation."""
    profile = state["profile"]

    # Calculate available investable surplus directly
    surplus = max(0.0, profile.income - profile.expenses)

    # Re-engineered prompt focusing explicitly on Indian investment vehicles like SIPs and Mutual funds
    prompt = f"""
    You are an expert personal finance advisor for college students and young adults in India.
    Based on the profile below, calculate their monthly investable surplus (Income: ₹{profile.income} - Expenses: ₹{profile.expenses} = ₹{surplus}).

    Provide a highly practical, 2-3 sentence recommendation of exactly how they should split this ₹{surplus} surplus using highly relatable Indian options:
    - Emergency Cash (High-yield Savings Account or Liquid Mutual Fund)
    - Monthly Systematic Investment Plans (SIPs) in Equity/Index Mutual Funds for growth
    - Safe avenues like PPF or Fixed Deposits if their goal is short-term/low-risk.

    Match the breakdown perfectly to their specific investment reason: "{profile.reason_for_investment}".
    End the response by asking clearly: "Does this breakdown look good to you?"
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"initial_recommendation": response.content}

# ==========================================
# 3. ROUTING & CONDITIONAL LOGIC
# ==========================================

def route_after_gathering(state: AgentState):
    if state["current_step"] == "VERIFICATION":
        return "generate_recommendation"
    return "ask_more_questions"

# ==========================================
# 4. BUILD AND COMPILE THE GRAPH
# ==========================================

workflow = StateGraph(AgentState)

workflow.add_node("gather_info", gather_info_node)
workflow.add_node("recommend", recommend_node)

workflow.set_entry_point("gather_info")

workflow.add_conditional_edges(
    "gather_info",
    route_after_gathering,
    {
        "ask_more_questions": END,
        "generate_recommendation": "recommend"
    }
)
workflow.add_edge("recommend", END)
app = workflow.compile()

# ==========================================
# 5. LIVE INTERACTIVE NOTEBOOK CHAT LOOP
# ==========================================

def start_colab_agent():
    # Tailored to prompt for Indian context metrics
    onboarding_guidance = (
        "Hello! Let's build your practical, student-friendly investment blueprint.\n"
        "To get started, please tell me:\n"
        " 1. Your monthly income/allowance (e.g., 'My income is 15000')\n"
        " 2. Your monthly expenses (e.g., 'I spend 7000')\n"
        " 3. What you are saving or investing for (e.g., 'Buying a smartphone next year' or 'Starting an SIP for long-term growth')\n\n"
        "What are your current income and expense numbers?"
    )

    state = {
        "messages": [AIMessage(content=onboarding_guidance)],
        "profile": FinancialProfile(),
        "initial_recommendation": "",
        "current_step": "GATHERING"
    }

    print(f"🤖 Agent:\n{state['messages'][0].content}")

    while True:
        user_input = input("\n🧑 User: ")
        if user_input.lower() in ['exit', 'quit']:
            print("🤖 Agent: Goodbye!")
            break

        state["messages"].append(HumanMessage(content=user_input))

        if state["current_step"] == "GATHERING":
            output = app.invoke(state)
            state.update(output)

            if state["profile"].missing_question:
                bot_msg = state["profile"].missing_question
                print(f"\n🤖 Agent: {bot_msg}")
                state["messages"].append(AIMessage(content=bot_msg))
            else:
                bot_msg = state["initial_recommendation"]
                print(f"\n🤖 Agent:\n{bot_msg}")
                state["messages"].append(AIMessage(content=bot_msg))

        elif state["current_step"] == "VERIFICATION":
            check_prompt = f"Analyze if this user response means they accept or agree with the plan: '{user_input}'. Respond strictly with only the word 'YES' or 'NO'."
            verification = llm.invoke([HumanMessage(content=check_prompt)]).content.strip().upper()

            if "YES" in verification:
                # Step 6: Upgraded to guide them through mobile app implementation blueprints
                blueprint_prompt = f"""
                The user approved your allocation strategy! Now provide a highly practical, relatable, step-by-step execution roadmap for a beginner in India.

                Based on your strategy ({state['initial_recommendation']}), explain exactly:
                1. **Where to keep their Emergency Fund:** (e.g., keeping it in a sweep-in Fixed Deposit or a liquid mutual fund via an app so it's instantly usable).
                2. **How to start their Wealth Investment:** Explain how to set up a monthly **SIP** using user-friendly apps (like Groww, Coin by Zerodha, or Dhan). Suggest a relatable fund type (like a simple Nifty 50 Index Fund for low-cost broad market exposure).
                3. **Milestone tracking:** Give a realistic estimation based on their reason: "{state['profile'].reason_for_investment}".

                Keep it completely action-focused, jargon-free, and practical for someone managing money directly on their smartphone.
                """
                detailed_plan = llm.invoke([HumanMessage(content=blueprint_prompt)]).content
                print(f"\n🏁 FINAL PRACTICAL ACTION PLAN:\n{detailed_plan}")
                state["current_step"] = "COMPLETED"
                break
            else:
                state["current_step"] = "GATHERING"
                state["initial_recommendation"] = ""
                reset_msg = "No problem at all! Let's adjust. What changes should we make to your investment targets or layout?"
                print(f"\n🤖 Agent: {reset_msg}")
                state["messages"].append(AIMessage(content=reset_msg))

# Start the interactive assistant
start_colab_agent()

🤖 Agent:
Hello! Let's build your practical, student-friendly investment blueprint.
To get started, please tell me:
 1. Your monthly income/allowance (e.g., 'My income is 15000')
 2. Your monthly expenses (e.g., 'I spend 7000')
 3. What you are saving or investing for (e.g., 'Buying a smartphone next year' or 'Starting an SIP for long-term growth')

What are your current income and expense numbers?

🧑 User: My income is 30000. And my expenses are 20000

🤖 Agent: What is the core goal or reason for wanting to invest?

🧑 User: savings

🤖 Agent: What is the core goal or reason for wanting to invest?

🧑 User: saving

🤖 Agent:
Based on the profile, the monthly investable surplus is ₹10000.0. For someone focused on "saving", I recommend allocating 50% (₹5000.0) towards Emergency Cash in a High-yield Savings Account or Liquid Mutual Fund for easy access, 30% (₹3000.0) towards Safe avenues like PPF for stable, low-risk returns, and 20% (₹2000.0) towards Monthly Systematic Investment Plans (SIPs